# Colour Brings People Together
### Part B · Auditing the UN SDG palette across languages, colour vision and age

The United Nations identifies its 17 Sustainable Development Goals with 17 colours, used unchanged in every country and every language. In the next 45 minutes you and your partner will test how well that palette communicates along two axes:

| Axis | Question | Assessed |
|---|---|---|
| **Perceptual** | Can people still tell the colours apart with a colour vision deficiency, or with an ageing eye? | within a community |
| **Cognitive** | Do people reliably agree on what each colour is called, and do the names in different languages point to the same colour? | across languages |

![The Colour Spinner](https://raw.githubusercontent.com/akvilesinkeviciute/colour-brings-people-together/v1.0/images/colour_spinner.png)

*The Colour Spinner's six linked views: Picker, Namer, Thesaurus, Translator, Mapper and Profiler, with controls for language and for colour vision deficiency and ageing simulations.*

A palette can succeed on one axis and fail on the other. By the end you will have a two-axis picture of a subset of SDG colours, your **test colours**, and a comparison with what an AI system says the same colours are called.

**How to use this notebook.** Run each cell in order with **Shift + Enter**. The only things you need to change are the grey form fields on the right of some cells. Keep the [Colour Spinner](https://colornaming.net/spinner) open in another tab.

In [ ]:
#@title Run me first: load the masterclass toolkit
#@markdown The colour maths, the Colour Spinner connection and the charts live in `colour_masterclass.py`, published next to these notebooks.
TOOLKIT_URL = "https://raw.githubusercontent.com/akvilesinkeviciute/colour-brings-people-together/v1.0/colour_masterclass.py"  #@param {type:"string"}
import os, urllib.request
if not os.path.exists("colour_masterclass.py"):
    try:
        urllib.request.urlretrieve(TOOLKIT_URL, "colour_masterclass.py")
    except Exception as e:
        raise RuntimeError("Could not download the toolkit. Check your internet connection, or upload "
                           "colour_masterclass.py in Colab's Files panel (folder icon, left).") from e
from colour_masterclass import *

## 1 · Meet the palette

Every swatch in this notebook sits on a neutral mid-grey surround. A white or black page changes how a colour looks, so colour scientists judge colours against grey.

In [ ]:
show_palette(SDG.hex.tolist(), labels=SDG.index.tolist(), captions=SDG.goal.tolist(),
             note="The 17 UN Sustainable Development Goal colours (United Nations 2015).")

## 2 · Choose your subset and languages

Pick **three to five** SDG colours. Good audits mix an easy colour with a hard one: try one of the reds (1, 4, 5, 8), one of the blues (6, 14, 16, 17) and something in between such as 12.

Choose a **source language**, then tick the **target languages** to compare it with. The Colour Spinner currently covers six. If either of you speaks one of them natively, include it, and include **OtjiHimba**, spoken by the Himba of northern Namibia, which has just seven indispensable colour terms: *serandu*, for example, covers what English splits into red, pink and orange.

**Backend.** *Colour Spinner API* fetches everything for you. If the network is slow, choose *Copied from the browser* and fill in the sheet in section 4.

In [ ]:
#@title Your pair's choices { run: "auto" }
SUBSET = "1, 6, 12, 14"  #@param {type:"string"}
SOURCE_LANGUAGE = "British English"  #@param ["American English", "British English", "French", "Greek", "OtjiHimba", "Spanish"]
#@markdown **Target languages** (tick any):
American_English = False  #@param {type:"boolean"}
British_English = True  #@param {type:"boolean"}
French = False  #@param {type:"boolean"}
Greek = True  #@param {type:"boolean"}
OtjiHimba = True  #@param {type:"boolean"}
Spanish = True  #@param {type:"boolean"}
BACKEND = "Colour Spinner API"  #@param ["Colour Spinner API", "Copied from the browser"]

ticked = {"American English": American_English, "British English": British_English,
          "French": French, "Greek": Greek, "OtjiHimba": OtjiHimba, "Spanish": Spanish}
LANGUAGES = pick_languages(SOURCE_LANGUAGE, ticked)
PICK = parse_subset(SUBSET)
show_subset(PICK, LANGUAGES)

## 3 · The perceptual axis: can everyone tell them apart?

About 1 in 12 men and 1 in 200 women of Northern European descent have a colour vision deficiency, most often a reduced ability to separate reds from greens. Independently, the lens of the eye yellows with age and absorbs more short-wavelength light, so blues and greens drift towards one another.

The rows below show the whole palette under each condition, simulated by the Colour Spinner's own models, the same ones you see with its "Simulate age/CVD" menu: protanopia, deuteranopia and tritanopia, and lens yellowing at ages 40, 60 and 80. Look down each column for colours that change, and across each row for colours that merge.

These simulations show how a colour might *appear*, not how people with a colour vision deficiency or an older eye would *name* it: the Spinner's naming models were trained on people with typical colour vision.

In [ ]:
prefetch_simulations(SDG.hex.tolist())      # ask the Colour Spinner for all 17 colours at once
show_conditions(SDG.hex.tolist(), labels=[str(i) for i in SDG.index])

The chart below makes "merge" measurable. For each colour in your subset it finds the **nearest other SDG colour** under each condition and reports the CIEDE2000 colour difference (ΔE00) between them. A difference of about 1 is just noticeable side by side; for colours that must be told apart at a glance in a chart or a sign, aim for 10 or more. The small number under each value names the nearest neighbour.

**A note on the distances.** All colour differences in this notebook are CIEDE2000 (ΔE00). CIEDE2000 was derived from colour-difference judgements by observers with typical colour vision. Applied to simulated colours, it estimates how different those colours would look to a typical observer, which only approximates what people with a colour vision deficiency or an older eye can tell apart.

In [ ]:
#@title Nearest-neighbour distinctiveness { run: "auto" }
CONFUSABLE_BELOW_DELTA_E = 10  #@param {type:"slider", min:3, max:25, step:1}
#@markdown Two colours closer than this CIEDE2000 colour difference count as easily confused. About 1 is just noticeable side by side; 10 is a common rule of thumb for telling colours apart at a glance.

nn_de, nn_id = nearest_neighbours(PICK)      # ΔE00 to the nearest other SDG colour, per condition
plot_nearest_neighbours(nn_de, nn_id, CONFUSABLE_BELOW_DELTA_E)
show_at_risk(nn_de, nn_id, CONFUSABLE_BELOW_DELTA_E)

## 4 · The cognitive axis: what are these colours called?

Here you ask the Colour Spinner what speakers of each language actually call your colours. Its naming model, Spin Colour Forest, is trained on about 81,000 free colour names from 2,450 speakers, collected online at [colornaming.net](https://colornaming.net) and, for OtjiHimba, in fieldwork in Namibia. Treat it as a reference baseline rather than ground truth: it covers six languages, and in each only the **indispensable vocabulary**, the minimal set of terms needed to name every colour. That ranges from 7 terms in OtjiHimba to 47 in American English (32 in British English, Greek and Spanish; 27 in French).

Two measures come out of this step:

- **Nameability**: the Namer's *confidence*, the probability of its top-ranked name in the model's naming distribution for that colour. High confidence marks the centre of a category, where speakers agree; low confidence marks a contested boundary between categories.
- **Translatability**: whether each language's name points to the same colour. The Mapper's *best example* of each name, its **focal colour** (the top-left tile), shows where that name is centred; if the focal colours drift apart across languages, a "translation" of the colour lands somewhere else.

**Only if you chose *Copied from the browser*:** fill in the sheet below, one row per colour and language. In the Spinner, type the hex into the Picker, set the Namer to the language, and copy the name, its confidence and the RGB of the Mapper's best example (the focal colour). Otherwise skip this cell.

In [ ]:
SHEET = """
hex, language, name, confidence, best_example
# C34F4F, American English, Red, 20%, 232 21 56      ← example row, as the Spinner shows #C34F4F
"""

In [ ]:
spinner = connect(BACKEND, SHEET)

rows, problems = [], []
for sdg in PICK:
    for language in LANGUAGES:
        colour = SDG.hex[sdg]
        try:
            namer = spinner.name(colour, language)                  # Namer: top name and its confidence
        except Exception as e:
            problems.append(f"SDG {sdg} in {language}: {e}"); continue
        best = safe(spinner.best_example, namer["name"], language)  # Mapper: where that name is centred
        rows.append({"sdg": sdg, "hex": colour, "language": language,
                     "name": namer["name"], "romanized": namer.get("romanized"),   # Latin spelling, e.g. kokkino
                     "confidence": namer["confidence"], "best_example": best,
                     "drift": de00(colour, best) if best else np.nan})  # how far the name pulls the colour
naming = pd.DataFrame(rows)

report_problems(problems)
show_names_table(naming, LANGUAGES)

### Nameability

Each marker is one language. Colours near the right have a confident top name; colours on the left are where the Spinner is less certain what speakers would call them, and where a label such as "the red goal" will mean different things to different readers.

In [ ]:
plot_nameability(naming, PICK, LANGUAGES)

### Translatability

The left swatch is the test colour. Each swatch to its right is the **best example** of the name that language uses for it, with its distance from the test colour. When those swatches differ from each other, the "same" colour word does not mean the same colour in each language.

In [ ]:
show_translatability(naming, PICK, LANGUAGES, CONFUSABLE_BELOW_DELTA_E)

The same information as a map. This is a top-down view of CAM16-UCS colour space, the perceptually uniform space the Colour Spinner works in: hue runs around the centre and colourfulness increases outwards. Each large disc is a test colour; the lines lead to where each language's name for it is centred.

In [ ]:
plot_name_map(naming, PICK, LANGUAGES)

## 5 · Two axes, one picture

Each disc is a test colour. Across: its **perceptual distance**, the ΔE00 to the nearest other SDG colour, for typical colour vision unless you choose another condition in the form. Up: its **naming confidence**, averaged over your languages unless you choose one. Both axes run from 0 to 1, so the two are on the same scale. The grey circle marks where the two thresholds meet. The table underneath gives the numbers, with each language's name and confidence.

In [ ]:
#@title Scorecard { run: "auto" }
SCORE_LANGUAGE = "Average of my languages"  #@param ["Average of my languages", "American English", "British English", "French", "Greek", "OtjiHimba", "Spanish"]
SCORE_CONDITION = "Typical"  #@param ["Typical", "Protanopia", "Deuteranopia", "Tritanopia", "Age 40", "Age 60", "Age 80", "Worst case of all conditions"]
NORMALISE_BY = "Largest value in my subset"  #@param ["Largest value in my subset", "Range across all 17 SDG colours"]
HARD_TO_NAME_BELOW_CONFIDENCE = 0.5  #@param {type:"slider", min:0.2, max:0.9, step:0.05}
#@markdown **SCORE_LANGUAGE**: whose naming confidence to plot. **SCORE_CONDITION**: whose vision the perceptual distance is for (*Typical* = typical colour vision).
#@markdown A colour whose top name has a Spinner confidence below **HARD_TO_NAME_BELOW_CONFIDENCE** counts as hard to name.

card = scorecard(PICK, naming, SCORE_LANGUAGE, SCORE_CONDITION, NORMALISE_BY)
plot_two_axes(card, CONFUSABLE_BELOW_DELTA_E, HARD_TO_NAME_BELOW_CONFIDENCE)
show_scorecard(card)

## 6 · What does an AI system say?

Now ask a current AI system to name the same colours in the same languages, and compare its answers with how speakers actually name them.

**Which AI?** Use the AI you use day to day (ChatGPT, Claude, Gemini, Copilot, …): the comparison is most meaningful that way, and it's what the boxes below are for. Colab's built-in Gemini is also offered, for demonstration.

Choose in the form below:

- **Colab's Gemini:** choose it in the form and run the cells in order. It's for demonstration and stops working once Colab's free usage limit is reached.
- **Your own AI:** keep *Paste answers from my own AI*. The next cell shows the prompt to copy into your AI tool, and one box per answer with the colour codes already filled in: you only copy in the names.

In [ ]:
#@title Which AI?
AI_PROVIDER = "Paste answers from my own AI"  #@param ["Paste answers from my own AI", "Gemini built into Colab"]
#@markdown **Your own AI** (ChatGPT, Claude, Gemini, Copilot, …): the next cell shows a prompt to copy into it and one box per colour and language. The colour codes are filled in for you; just copy in the names your AI gives.
#@markdown **Gemini built into Colab** is for demonstration. It needs no set-up but stops working once Colab's free usage limit is reached.

In [ ]:
lines = "\n".join(f"{r.hex} | {r.language}" for _, r in naming.iterrows())
himba = "\n(OtjiHimba is spoken by the Himba of northern Namibia.)" if "OtjiHimba" in LANGUAGES else ""
PROMPT_B = f"""You are helping with a cross-cultural colour naming study.
For each colour code and language below, give the single colour name that most native speakers of that
language would use for it. Write each name in that language and in its usual script; for a non-Latin
script, add the Latin spelling in brackets.{himba}

{lines}

Reply with exactly one line for each, in this format and nothing else:
#E5243B | {LANGUAGES[-1]} | name"""
show_prompt(PROMPT_B, AI_PROVIDER, rows=naming[["hex", "language"]], kind="name")

In [ ]:
if AI_PROVIDER == PASTE:
    ai_names = collected_answers()                        # the names typed into the boxes
else:
    ai_names = read_ai_names(ask_ai(PROMPT_B, AI_PROVIDER))
ai_names["language"] = ai_names["language"].map(canonical_language)

comparison = naming.merge(ai_names, on=["hex", "language"], how="left")
comparison["agrees"] = [same_name(ai, name, rom)    # any script; ignores case and accents
                        for ai, name, rom in zip(comparison.ai_name, comparison.name, comparison.romanized)]
comparison = where_ai_names_point(comparison, spinner)   # Mapper: where each differing AI name is centred
show_name_comparison(comparison, CONFUSABLE_BELOW_DELTA_E)
plot_agreement_by_language(comparison, LANGUAGES)

A match on a colour with low confidence says little: when the Spinner's confidence in its top name is only around 30%, other names are also defensible. So for every name that differs, the last column asks the Spinner's Mapper where the AI's name is centred for speakers of that language. If that colour is close to the test colour, the AI chose a plausible neighbouring name. If it is far away, or the name isn't one speakers use at all, the AI's name would mislead them. Large distances on high-agreement colours are the ones to bring to the group.

## 7 · Save your results for Part C

Part C picks up the names you found here. Run the next cell to download `partB_results.csv`. If you are working in Colab, keep the tab open: Part C can also reload it from your Downloads.

In [ ]:
out = naming.merge(ai_names, on=["hex", "language"], how="left") if "ai_names" in globals() else naming
out.to_csv("partB_results.csv", index=False)
download("partB_results.csv")

## Before you share with the room

Agree with your partner on one sentence for each:

1. Which colour performs best on both axes, and which worst?
2. Is your worst colour failing **perceptually**, **cross-linguistically** or **both**?
3. Where did the AI system's names diverge from how speakers name the colour, and does the divergence look like a default towards English or "universal" colour categories?